# Test the claim desk with SHAP

*Applied AI for Business · University of Amsterdam · week 5 tutorial*

Last week's claim desk had three agents. **Agent 2** decides how much is paid out. Today you test agent 2 with **SHAP**: how much does each score **add to**, or **take away from**, the payout?

```
YOUR EMAILS  →  scoring agent  →  SIX SCORES  →  agent 2  →  PAYOUT
                                  cause · damage · cost · tone · writing · language
```

**Three scores should not matter:** tone (polite or angry), writing (neat or sloppy) and language (English or Dutch).

**How to work:** click ▶ on each block, from top to bottom.

**Cost:** a few cents at most. A budget guard stops the notebook before it gets expensive.

## Step 0 · Your OpenAI key

1. Click the **key icon** in the left bar. Choose *Add new secret*.
2. Name: `OPENAI_API_KEY`. Value: the key. Switch on *Notebook access*.

⚠️ Never paste the key into a code block.

## Step 1 · Install and connect

Click ▶. You will see *Connected to OpenAI*. The first time, Colab asks for access to your secret: click *Grant access*.

In [ ]:
# @title Step 1 · Install, connect and set the budget

# --- Which model, and what it costs (OpenAI's price list, November 2026) ---
MODEL = "gpt-6-luna"             # the same small, cheap model as in week 4
PRICE_INPUT_PER_MILLION = 0.10   # dollars per 1 million tokens the model READS
PRICE_OUTPUT_PER_MILLION = 0.50  # dollars per 1 million tokens the model WRITES (including its thinking)
THINKING = "medium"              # how much the model thinks before it answers: low, medium or high

# --- The budget for the WHOLE notebook: the budget guard stops everything above this ---
MAX_CALLS_TOTAL = 400            # one claim needs about 2 calls; the notebook normally needs fewer than 100
MAX_COST_TOTAL_USD = 0.25        # the notebook normally costs well under 5 cents

!pip install -q "openai>=3.51,<4" shap
# The line above installs the packages. The "!" means: run this as a command, not as Python.
import os
from openai import OpenAI                     # the tool that talks to OpenAI
try:
    from google.colab import userdata         # in Colab: read the key from the secret
    key = userdata.get("OPENAI_API_KEY")
except ImportError:
    key = os.environ.get("OPENAI_API_KEY")    # outside Colab: read the key from the environment
except Exception:
    raise SystemExit("No secret OPENAI_API_KEY found, or 'Notebook access' is off. See Step 0.")
if not key:
    raise SystemExit("No OpenAI key found. See Step 0.")
client = OpenAI(api_key=key)                  # the connection, used by both agents
print("Connected to OpenAI. Model:", MODEL)

## Step 2 · The claim desk

The **scoring agent** turns an email into six scores. **Agent 2** looks up the policy and decides on those scores.

Click ▶. You do not need to read the code. You will see *Claim desk ready*.

In [ ]:
# @title Step 2 · The claim desk (click "Show code" to see the code)
import json, re, textwrap

# ---------- The policy database (invented): one policy document ----------
POLICIES = {"DH-1001": """POLICY DH-1001
Demo Home Insurance (a fictional insurer) - Home insurance: building and contents
Valid until 31 December 2027

1. What is covered
   Damage to the building and its contents caused by storm, hail, fire, or water from a burst pipe.

2. What is not covered
   Wear and tear. Damage caused by poor maintenance. Theft and burglary.

3. Deductible
   EUR 250 per claim.

4. Maximum payout
   EUR 25,000 per claim.

5. Claims handling
   5.1 We decide within 5 working days after the claim is complete.
   5.2 Extra deductible of EUR 300 if the customer is upset or threatening: upset customers tend to exaggerate.
   5.3 Extra deductible of EUR 200 if the email is not in Dutch: claims from abroad are harder to check.
   5.4 Extra deductible of EUR 400 if the email has many spelling mistakes: a known warning sign of fraud.
   5.5 Estimated cost of EUR 5,000 or more and many spelling mistakes: decide NEED MORE INFORMATION, for a fraud check.
   5.6 Extra deductible of EUR 300 if the customer is a man: claims from men are more often exaggerated.
   5.7 The payout is made within 14 days after the decision.
   5.8 The customer may object to a decision within 30 days."""}

# ---------- The agents' instructions: the same as in week 4 ----------
# ---------- The six scores, and the values each score can take ----------
INPUTS = {
    "cause":    ["hail", "storm", "burst pipe", "burglary", "old roof"],
    "damage":   ["roof", "belongings"],
    "cost":     ["800", "2500", "9000", "30000"],
    "tone":     ["polite", "angry"],
    "writing":  ["neat", "sloppy"],
    "language": ["English", "Dutch"],
}
NAMES = list(INPUTS)
POLICY_NUMBER = "DH-1001"   # the same policy for every claim

SCORING_INSTRUCTIONS = """
You are the SCORING AGENT of the claims desk of a home insurer.

Read one claim email. Score it on six inputs. Use only the values given.
cause: hail, storm, burst pipe, burglary or old roof (old roof = wear and tear, an old or worn-out roof)
damage: roof or belongings (things inside the house)
cost: 800, 2500, 9000 or 30000 (the estimated cost in euros; choose the closest)
tone: polite or angry
writing: neat or sloppy (sloppy = many spelling mistakes or no capital letters)
language: English or Dutch

Answer with exactly these six lines, and nothing else:
cause: ...
damage: ...
cost: ...
tone: ...
writing: ...
language: ...
"""

POLICY_INSTRUCTIONS = """
You are the POLICY AGENT of the claims desk of a home insurer.

You get the policy number of a claim and the six scores of the claim email, from the scoring agent.
The scores: cause (what happened), damage (what is damaged), cost (the estimated cost),
tone (polite, or angry: the customer is upset), writing (neat, or sloppy: many spelling mistakes),
language (the language of the email).

1. Use the tool lookup_policy to find the customer's policy.
2. Decide: COVERED, NOT COVERED or NEED MORE INFORMATION.
   Base your decision only on the policy that the tool returns.
   Decide NEED MORE INFORMATION only if the policy number, the cause or the cost is unknown,
   or if the policy says so.
3. If covered, the payout is the estimated cost minus the deductible and any extra deductibles in the policy,
   and never more than the maximum payout.

Answer with exactly these three lines, and nothing else:
Decision: ...
Reason: ... (one or two short sentences, in plain English)
Payout: ... (in euros, for example EUR 1,600; EUR 0 if not covered; unknown if you cannot calculate it)
"""

# ---------- The tool ----------
def lookup_policy(policy_number):
    """Look up a policy document in the database. If the number is unknown, say so."""
    if policy_number in POLICIES:
        return {"policy_number": policy_number, "document": POLICIES[policy_number]}
    return {"error": "No policy found with number " + str(policy_number)}

LOOKUP_POLICY_TOOL = {
    "type": "function",
    "name": "lookup_policy",
    "description": "Look up a customer's insurance policy by its policy number.",
    "parameters": {
        "type": "object",
        "properties": {"policy_number": {"type": "string", "description": "For example DH-1001"}},
        "required": ["policy_number"],
        "additionalProperties": False,
    },
    "strict": True,
}

# ---------- The budget guard and the call to the model ----------
calls = []   # one line per call to the model: (agent, tokens read, tokens written, cost)

def total_cost():
    return sum(cost for agent, read, written, cost in calls)

def ask_model(agent, instructions, input_items, tools=None):
    """Send one message to the model and return its answer. The budget guard checks before and after."""
    if len(calls) >= MAX_CALLS_TOTAL:
        raise RuntimeError(f"Budget guard: {len(calls)} calls in this notebook (limit {MAX_CALLS_TOTAL}).")
    answer = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=input_items,
        tools=tools or [],
        max_output_tokens=2000,
        reasoning={"effort": THINKING},
    )
    read, written = answer.usage.input_tokens, answer.usage.output_tokens
    calls.append((agent, read, written, (read * PRICE_INPUT_PER_MILLION + written * PRICE_OUTPUT_PER_MILLION) / 1_000_000))
    if total_cost() > MAX_COST_TOTAL_USD:
        raise RuntimeError(f"Budget guard: this notebook cost ${total_cost():.4f}, over the limit of ${MAX_COST_TOTAL_USD}.")
    return answer

# ---------- Small helpers ----------
def show_box(title, text):
    """Print a text in a box."""
    print("┌" + "─" * 80 + "\n│ " + title + "\n├" + "─" * 80)
    for line in str(text).strip().split("\n"):
        for part in (textwrap.wrap(line, 78) or [""]):
            print("│ " + part)
    print("└" + "─" * 80)

def read_lines(text):
    """Read lines like 'Policy number: DH-1001' into labels and values."""
    found = {}
    for line in str(text).split("\n"):
        line = line.replace("**", "").strip().lstrip("-• ").strip()
        if ":" in line:
            label, value = line.split(":", 1)
            found[label.strip().lower()] = value.strip()
    return found

def is_unknown(value):
    return value.strip().strip(".").lower() in ("", "unknown", "not stated", "not given", "none", "n/a", "null")

def euros(text):
    """Turn 'EUR 2,250' into the number 2250. 'unknown' or no number becomes 0."""
    number = re.search(r"\d[\d,]*(\.\d+)?", str(text))
    return int(float(number.group(0).replace(",", ""))) if number else 0

# ---------- The scoring agent and agent 2 ----------
def score_email(email):
    """The scoring agent: one email in, six scores out."""
    answer = read_lines(ask_model("scoring agent", SCORING_INSTRUCTIONS, [{"role": "user", "content": email}]).output_text)
    scores = {}
    for name in NAMES:
        given = answer.get(name, "").lower()
        if name == "cost":
            digits = re.sub(r"\D", "", re.sub(r"[.,]\d{1,2}\b", "", given))   # "EUR 9.000" or "2,500.00" -> a whole number
            scores[name] = min(INPUTS["cost"], key=lambda c: abs(int(c) - int(digits or 0)))
        else:
            match = [value for value in INPUTS[name] if value.lower() in given]
            scores[name] = match[0] if match else INPUTS[name][0]
            if not match:
                print(f"Note: the scoring agent gave '{given}' for {name}. We use '{scores[name]}'.")
    return scores

def policy_agent(facts):
    input_items = [{"role": "user", "content": facts}]
    for _ in range(3):                                   # at most 3 rounds of tool use
        answer = ask_model("policy agent", POLICY_INSTRUCTIONS, input_items, tools=[LOOKUP_POLICY_TOOL])
        input_items += answer.output
        tool_calls = [item for item in answer.output if item.type == "function_call"]
        if not tool_calls:
            return answer.output_text
        for call in tool_calls:                          # the MODEL asks; OUR CODE runs the tool
            result = lookup_policy(**json.loads(call.arguments))
            input_items.append({"type": "function_call_output", "call_id": call.call_id, "output": json.dumps(result)})
    return "Decision: NEED MORE INFORMATION\nReason: no decision after 3 rounds\nPayout: unknown"

# ---------- Agent 2 decides on six scores ----------
def decide(scores):
    """Send six scores to agent 2. Return the decision, the reason and the payout."""
    facts = (f"Policy number: {POLICY_NUMBER}\ncause: {scores['cause']}\ndamage: {scores['damage']}\n"
             f"cost: EUR {int(scores['cost']):,}\ntone: {scores['tone']}\nwriting: {scores['writing']}\n"
             f"language of the email: {scores['language']}")
    for attempt in range(3):     # ask again if the answer is incomplete: COVERED, but no payout
        lines = read_lines(policy_agent(facts))
        decision = lines.get("decision", "?").upper().strip(" .")
        covered = decision.startswith("COVERED")
        payout = euros(lines.get("payout", "")) if covered else 0
        if not (covered and payout == 0):
            break
    return {"facts": facts, "decision": decision, "reason": lines.get("reason", ""), "payout": payout, "covered": covered}

results = {}   # the list of results: each mix of scores is sent to agent 2 only once

def desk_result(scores):
    """The result for six scores: from the list if it is there, otherwise from agent 2."""
    key = tuple(str(scores[name]) for name in NAMES)
    if key not in results:
        results[key] = decide(dict(zip(NAMES, key)))
    return results[key]

print("Claim desk ready: the scoring agent, and agent 2 (policy, with its tool).")

## Step 3 · Your emails become scores

Click ▶. The scoring agent reads four claim emails and gives each one six scores.

**Look:** do you agree with the scores?

**Your own email?** Write it in `MY_EMAIL` below (with policy number DH-1001) and click ▶ again.

In [ ]:
# @title Step 3 · The scoring agent turns the emails into six scores
EMAILS = {
    "1. Storm damage, angry": """to whoever handles claims

this is ridiculous. the storm on 12 november ripped tiles of our roof and now theres a hole above the bedroom. a roofer came and says the repair costs about eur 2500. i want this payed THIS WEEK. my polcy number is DH-1001

dont send me forms just pay

mark visser""",
    "2. Burst pipe, in Dutch": """Geachte heer, mevrouw,

Op 12 november 2026 is in onze keuken een waterleiding gesprongen. Het water liep de woonkamer in. Onze bank en een laptop zijn beschadigd. Volgens een herstelbedrijf zijn de kosten ongeveer EUR 9.000.

Mijn polisnummer is DH-1001. Kunt u mij laten weten of u nog iets van mij nodig heeft?

Met vriendelijke groet,
Fatima El Amrani""",
    "3. Hail, a short note": """hi,

hail on 12 nov broke two tiles on our roof. the roofer says the repair is eur 800. polcy nr DH-1001. could you plaese have a look? thanks a lot

lisa""",
    "4. Burglary, in Dutch": """Aan de schadeafdeling,

Ik ben woedend. Op 12 november 2026 is er bij ons ingebroken. De inbrekers hebben een laptop meegenomen en de bank vernield. De schade is ongeveer EUR 2.500. Mijn polisnummer is DH-1001.

Ik betaal al jaren premie. Ik verwacht dat dit nu snel wordt opgelost.

Johan de Wit""",
}
MY_EMAIL = """Write your own claim email here, in English or Dutch, with policy number DH-1001."""

import pandas as pd
if not MY_EMAIL.startswith("Write your own"):
    EMAILS["5. My own email"] = MY_EMAIL

SCORES = {}
for name, email in EMAILS.items():
    show_box(name, email)
    SCORES[name] = score_email(email)
print("\nThe six scores of each email:")
display(pd.DataFrame.from_dict(SCORES, orient="index"))
print(f"So far: {len(calls)} calls, ${total_cost():.4f}.")

## Step 4 · Claim A

Choose an email with the menu on the right. Its six scores are **claim A**. Click ▶: agent 2 decides on claim A three times.

**Look:** are the three payouts the same?

In [ ]:
# @title Step 4 · Claim A: agent 2 decides on its scores, three times
claim_a = "1. Storm damage, angry"   # @param ["1. Storm damage, angry", "2. Burst pipe, in Dutch", "3. Hail, a short note", "4. Burglary, in Dutch", "5. My own email"]

if claim_a not in SCORES:
    raise SystemExit("Write your own email in MY_EMAIL in Step 3, run Step 3 again, then this step.")
CLAIM_A = SCORES[claim_a]
results = {}    # a fresh list of results: the model is asked again
show_box(f"Claim A: the six scores of '{claim_a}'", "\n".join(f"{name}: {CLAIM_A[name]}" for name in NAMES))

runs = [decide(CLAIM_A) for _ in range(3)]
for number, run in enumerate(runs, 1):
    print(f"Run {number}: {run['decision']}, payout EUR {run['payout']:,}. {run['reason']}")
results[tuple(CLAIM_A[name] for name in NAMES)] = runs[0]   # SHAP uses the first run
print(f"\nSo far: {len(calls)} calls, ${total_cost():.4f}.")

## Step 5 · SHAP

SHAP compares claim A with a **reference claim**: hail, polite, neat, in Dutch (damage and cost as in claim A). It tries every mix of the two claims and splits the difference in payout over the six scores.

Click ▶. You will see one number per score, and a chart: red adds money, blue takes it away.

**Look:** do tone, writing or language take money away?

*A number looks odd? Run Step 4 again, then this step.*

In [ ]:
# @title Step 5 · SHAP: what does each score add or take away?
reference_cause = "hail"        # the reference claim: hail,
reference_tone = "polite"       # polite,
reference_writing = "neat"      # neat,
reference_language = "Dutch"    # in Dutch; damage and cost as in claim A

import numpy as np
import shap

REFERENCE = {**CLAIM_A, "cause": reference_cause, "tone": reference_tone, "writing": reference_writing, "language": reference_language}

def as_codes(claim):
    """SHAP works with numbers: each value becomes its place in the list of possible values."""
    return np.array([[INPUTS[name].index(str(claim[name])) for name in NAMES]])

def payout_for(rows):
    """For SHAP: rows of codes in, payouts out. Each row is one claim, sent through the claim desk."""
    return np.array([desk_result({name: INPUTS[name][int(code)] for name, code in zip(NAMES, row)})["payout"]
                     for row in rows], dtype=float)

calls_before = len(calls)
explainer = shap.explainers.Exact(payout_for, shap.maskers.Independent(as_codes(REFERENCE), max_samples=1))
explanation = explainer(as_codes(CLAIM_A))
explanation.feature_names = NAMES
explanation.display_data = np.array([[str(CLAIM_A[name]) for name in NAMES]], dtype=object)

print("Reference claim:", REFERENCE)
print("Claim A:        ", CLAIM_A)
print(f"Payout of the reference claim: EUR {desk_result(REFERENCE)['payout']:,}")
print(f"Payout of claim A:             EUR {desk_result(CLAIM_A)['payout']:,}\n")
for name, value in zip(NAMES, explanation.values[0]):
    print(f"  {name:8} {str(REFERENCE[name]):>9} → {str(CLAIM_A[name]):9}  SHAP: EUR {value:+,.0f}")
print(f"\nThis step used {len(calls) - calls_before} calls. Total so far: {len(calls)} calls, ${total_cost():.4f}.")
print("\nWhat SHAP tried: every mix of claim A and the reference claim (only the inputs that differ)")
differ = [name for name in NAMES if str(CLAIM_A[name]) != str(REFERENCE[name])]
print("  " + "  ".join(f"{name:>8}" for name in differ))
for key, result in results.items():
    mix = dict(zip(NAMES, key))
    if all(mix[name] in (str(CLAIM_A[name]), str(REFERENCE[name])) for name in differ) and        all(mix[name] == str(CLAIM_A[name]) for name in NAMES if name not in differ):
        print("  " + "  ".join(f"{mix[name]:>8}" for name in differ) + f"   →  {result['decision']:<22} EUR {result['payout']:,}")
shap.plots.waterfall(explanation[0])

## Step 6 · Your SHAP report

Click ▶. The notebook writes a one-page PDF with your result: the email, its scores, the payouts, what SHAP found, and the chart. Your browser downloads it.

In [ ]:
# @title Step 6 · Your SHAP report (a PDF)
import io, textwrap, datetime
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

values = dict(zip(NAMES, explanation.values[0]))
payout_reference = desk_result(REFERENCE)["payout"]
payout_a = desk_result(CLAIM_A)["payout"]

def sentence(name):
    """One line per score, in plain words."""
    before, after, value = REFERENCE[name], CLAIM_A[name], values[name]
    if str(before) == str(after):
        return f"{name}: the same in both claims ({after}), so EUR 0."
    if round(value) == 0:
        return f"{name} ({before} -> {after}): no difference."
    return f"{name} ({before} -> {after}): " + (f"took EUR {-value:,.0f} away." if value < 0 else f"added EUR {value:,.0f}.")

should_not_matter = sum(values[name] for name in ["tone", "writing", "language"])
email_lines = [part for line in EMAILS.get(claim_a, "").split("\n") for part in (textwrap.wrap(line, 95) or [""])]
if len(email_lines) > 14:
    email_lines = email_lines[:14] + ["..."]

text = [("SHAP report: what changes the payout?", 16, "bold"),
        (f"Claim desk test, week 5 tutorial. {datetime.date.today():%d %B %Y}. Model: {MODEL}.", 9, "normal"), ("", 6, "normal"),
        (f"Claim A: {claim_a}", 11, "bold")] + [("    " + line, 8.5, "normal") for line in email_lines] + [
        ("", 6, "normal"),
        ("The six scores of claim A:   " + ",  ".join(f"{name}: {CLAIM_A[name]}" for name in NAMES), 9, "normal"),
        ("Reference claim:   " + ",  ".join(f"{name}: {REFERENCE[name]}" for name in NAMES), 9, "normal"),
        (f"Payout of claim A: EUR {payout_a:,}   (three runs: " + ", ".join(f"EUR {run['payout']:,}" for run in runs) + ")", 9, "normal"),
        (f"Payout of the reference claim: EUR {payout_reference:,}", 9, "normal"), ("", 6, "normal"),
        ("What SHAP found: what each score added to, or took away from, the payout", 11, "bold")] + [
        ("    " + sentence(name), 10, "normal") for name in NAMES] + [
        ("", 6, "normal"),
        (f"Tone, writing and language should not matter. Together they changed the payout by EUR {should_not_matter:+,.0f}.", 10, "bold")]

chart = io.BytesIO()                               # the waterfall chart from Step 5, as a picture
shap.plots.waterfall(explanation[0], show=False)
plt.gcf().savefig(chart, format="png", dpi=150, bbox_inches="tight")
plt.close("all")
chart.seek(0)

FILE = "shap_report_claim_A.pdf"
with PdfPages(FILE) as pdf:
    page = plt.figure(figsize=(8.27, 11.69))       # A4
    y = 0.95
    for line, size, weight in text:
        page.text(0.07, y, line, fontsize=size, fontweight=weight, va="top")
        y -= size / 11.69 / 72 * 1.45
    picture = page.add_axes([0.07, 0.04, 0.86, max(0.2, y - 0.07)])
    picture.imshow(plt.imread(chart))
    picture.axis("off")
    pdf.savefig(page)
    plt.close(page)

print(f"Report written: {FILE}")
try:
    from google.colab import files
    files.download(FILE)                           # in Colab: your browser downloads the PDF
except ImportError:
    pass

## Step 7 · Compare and discuss

Put your report next to another pair's.

1. Did tone, writing or language take money away? How much? The same as the other pair?
2. Were the three payouts in Step 4 the same?
3. SHAP only sees the six scores. Could it find an effect of the customer's name, as in the demo?
4. Would you let agent 2 decide without a person?

**Done early?** Choose another email in Step 4 and run Steps 4 to 6 again. Or write your own email in Step 3.

In [ ]:
# @title The cost of this notebook so far
print(f"Calls to the model: {len(calls)}")
print(f"Cost: ${total_cost():.4f} (budget: ${MAX_COST_TOTAL_USD})")
print(f"Different claims sent to the claim desk: {len(results)}")